# Simulation/Rating matrix test

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
workspace_root = next((p for p in [cwd, *cwd.parents] if (p / 'core').is_dir()), None)
if workspace_root is None:
    raise RuntimeError("Nie znaleziono katalogu gĹ‚Ăłwnego repozytorium.")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle
from core.properties import (
    DryAirPropertyProvider,
    GasMixturePropertyProvider,
    GasMixtureSpec,
)
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.models.heat_balance import BalanceSideSpec


In [2]:
def c_to_k(value_c):
    return None if value_c is None else value_c + 273.15

def kgh_to_kgs(value):
    return None if value is None else value / 3600.0

wet_gas_components = {
    'N2': 0.6627,
    'O2': 0.1761,
    'H2O': 0.1612,
}
wet_gas_spec = GasMixtureSpec(
    components=wet_gas_components, basis='mole', backend='HEOS', imposed_phase='gas',
)

media = {
    'dry_air': {
        'label': 'suche powietrze',
        'provider': DryAirPropertyProvider(),
    },
    'wet_gas_mixture': {
        'label': 'mieszanina N2/O2/H2O (gaz)',
        'provider': GasMixturePropertyProvider(wet_gas_spec),
    },
}

# Optional tube-surface roughness (v0.5.6). None = unspecified/hydraulically
# smooth (preserves the exact pre-roughness result); a positive absolute
# value [m] uses the Colebrook-White rough-pipe friction factor on the
# inner surface. roughness_outer is stored on the geometry but is not yet
# used by any outside-side correlation.
# typical values: 
# aluminium, new 1.5e-6
# aluminium, used 0.03e-3
# copper, new 0.0015e-3
# copper, used 0.03e-3
# steel, new, rolling skin 0.045e-3 HTRI default
# steel, slighty rusty 0.4e-3
# steel, after long operation cleaned 0.2e-3
# steel, intensely incrusted 4.0e-3
roughness_inner = 0.2e-3
roughness_outer = None
tube = BareTube(
    D_i=34.8e-3, D_o=38.1e-3, length_total=2.2, length_effective=2.15, wall_k=50.0,
    roughness_inner=roughness_inner, roughness_outer=roughness_outer,
)
euler_provider = 'gaddis_gnielinski'
# euler_provider = 'zukauskas'
bundle = TubeBundle(
    tube=tube, n_rows=20, n_tubes_per_row=40,
    pitch_transverse=48e-3, pitch_longitudinal=48e-3,
    layout='inline', n_passes_tube=1, flow_arrangement='crossflow',
)
hx = BareTubeHeatExchanger(bundle)


## Cases Matrix

Use `mode='simulate'` to calculate achievable duty and outlet temperatures. `overdesign` derates the effective conductance using `UA_eff = UA / (1 + overdesign)`.

Use `mode='rate'` to evaluate a specified heat balance. Provide `Q_kW`, `effectiveness`, or one complete side with an outlet temperature. Use `None` for values solved by the model.

Select `dry_air` or `wet_gas_mixture` independently for each side. 

In [3]:
cases = [
    {
        'case': 'S01', 'mode': 'simulate', 'overdesign': 0.15,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 18_220.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': None,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_380.0, 'outside_T_in_C': 400.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'R01', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 18_220.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': None,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_380.0, 'outside_T_in_C': 400.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': 0.45,
    },
]

pd.DataFrame(cases)


,case,mode,overdesign,inside_medium,inside_m_dot_kg_h,inside_T_in_C,inside_T_out_C,outside_medium,outside_m_dot_kg_h,outside_T_in_C,outside_T_out_C,Q_kW,effectiveness
0,S01,simulate,0.15,dry_air,18220.0,30.0,None,wet_gas_mixture,28380.0,400.0,None,None,NaN
1,R01,rate,NaN,dry_air,18220.0,30.0,None,wet_gas_mixture,28380.0,400.0,None,None,0.45


In [4]:
def rating_kwargs(case):
    if case.get('Q_kW') is not None:
        return {'Q': case['Q_kW'] * 1e3}
    if case.get('effectiveness') is not None:
        return {'effectiveness': case['effectiveness']}
    return {}  # duty wynika z kompletnej strony bilansu


from core.heat_transfer.internal_flow import prandtl_number as inside_prandtl_number
from core.heat_transfer.outside_flow import prandtl_number as outside_prandtl_number


def provider_for_medium(medium_key):
    medium_cfg = media.get(medium_key)
    if medium_cfg is None or 'provider' not in medium_cfg:
        known = ', '.join(sorted(media.keys()))
        raise KeyError(f"Nieznane medium={medium_key!r}. Dost\u0119pne: {known}")
    return medium_cfg['provider']


def thermal_details_from_state(state, result):
    """Extract thermal diagnostics from a converged IterativeThermalState.

    Pure field mapping -- no physical calculation happens here. See
    core.heat_transfer.thermal_iteration.IterativeThermalState /
    ThermalIterationDiagnostics for how these values were computed by the
    production thermal-iteration path (the single source of truth, shared
    by both simulate() and rate()).
    """
    if state is None:
        return {
            'inside_bulk_temperature': float('nan'),
            'inside_wall_temperature': float('nan'),
            'inside_wall_temperature_mean': float('nan'),
            'inside_wall_temperature_min_estimate': result.inside_wall_temperature_min_estimate,
            'inside_wall_temperature_max_estimate': result.inside_wall_temperature_max_estimate,
            'outside_wall_temperature_mean': float('nan'),
            'outside_wall_temperature_min_estimate': result.outside_wall_temperature_min_estimate,
            'outside_wall_temperature_max_estimate': result.outside_wall_temperature_max_estimate,
            'inside_Nu_base': float('nan'),
            'inside_length_correction': float('nan'),
            'inside_wall_temperature_correction': float('nan'),
            'inside_Nu_corrected': float('nan'),
            'inside_alfa_base': float('nan'),
            'inside_alfa_corrected': float('nan'),
        }
    diag = state.diagnostics
    return {
        'inside_bulk_temperature': state.inside_bulk_temperature,
        'inside_wall_temperature': state.inside_wall_temperature,
        'inside_wall_temperature_mean': result.inside_wall_temperature_mean,
        'inside_wall_temperature_min_estimate': result.inside_wall_temperature_min_estimate,
        'inside_wall_temperature_max_estimate': result.inside_wall_temperature_max_estimate,
        'outside_wall_temperature_mean': result.outside_wall_temperature_mean,
        'outside_wall_temperature_min_estimate': result.outside_wall_temperature_min_estimate,
        'outside_wall_temperature_max_estimate': result.outside_wall_temperature_max_estimate,
        'inside_Nu_base': diag.inside_Nu_base,
        'inside_length_correction': diag.inside_length_correction,
        'inside_wall_temperature_correction': diag.inside_wall_temperature_correction,
        'inside_Nu_corrected': diag.inside_Nu_corrected,
        'inside_alfa_base': diag.inside_alfa_base,
        'inside_alfa_corrected': state.alfa_i,
    }


def wall_props_from_state(state):
    """Extract wall-state property columns from a converged IterativeThermalState.

    Also a pure extraction; the only derived quantity is Pr = cp*mu/k, using
    the project's own prandtl_number() helper (a property-ratio definition,
    not a heat-transfer correlation) on properties the production code
    already evaluated.
    """
    if state is None or state.inside_wall_props is None or state.outside_wall_props is None:
        return {
            'inside_wall_mu_Pa_s': float('nan'), 'inside_wall_k_W_mK': float('nan'),
            'inside_wall_cp_J_kgK': float('nan'), 'inside_wall_Pr': float('nan'),
            'outside_wall_mu_Pa_s': float('nan'), 'outside_wall_k_W_mK': float('nan'),
            'outside_wall_cp_J_kgK': float('nan'), 'outside_wall_Pr': float('nan'),
        }
    iw = state.inside_wall_props
    ow = state.outside_wall_props
    return {
        'inside_wall_mu_Pa_s': iw.mu, 'inside_wall_k_W_mK': iw.k, 'inside_wall_cp_J_kgK': iw.cp,
        'inside_wall_Pr': inside_prandtl_number(iw.cp, iw.mu, iw.k),
        'outside_wall_mu_Pa_s': ow.mu, 'outside_wall_k_W_mK': ow.k, 'outside_wall_cp_J_kgK': ow.cp,
        'outside_wall_Pr': outside_prandtl_number(ow.cp, ow.mu, ow.k),
    }


def tube_bundle_roughness_diagnostics(tube_bundle):
    """Extract roughness and friction-factor pressure-drop diagnostics
    directly from the production TubeBundleHydraulicResult.

    Pure field mapping -- no physical calculation happens here. See
    core.pressure_drop.internal_pressure_drop.TubeBundleHydraulicResult /
    core.pressure_drop.straight_sections for how these values were computed
    by the production pressure-drop path.
    """
    if tube_bundle is None:
        return {
            'inside_roughness_inner_mm': float('nan'),
            'inside_relative_roughness': float('nan'),
            'inside_friction_factor_in': float('nan'),
            'inside_friction_factor_mid': float('nan'),
            'inside_friction_factor_out': float('nan'),
            'inside_friction_factor_method_in': None,
            'inside_friction_factor_method_mid': None,
            'inside_friction_factor_method_out': None,
            'inside_dp_straight_tube_friction_Pa': float('nan'),
            'inside_dp_straight_tube_acceleration_Pa': float('nan'),
            'inside_dp_tube_entrances_Pa': float('nan'),
            'inside_dp_tube_exits_Pa': float('nan'),
            'inside_dp_tube_bundle_Pa': float('nan'),
        }
    return {
        'inside_roughness_inner_mm': (
            None if tube_bundle.roughness_inner is None else tube_bundle.roughness_inner * 1000.0
        ),
        'inside_relative_roughness': tube_bundle.relative_roughness_inner,
        'inside_friction_factor_in': tube_bundle.friction_factor_in,
        'inside_friction_factor_mid': tube_bundle.friction_factor_mid,
        'inside_friction_factor_out': tube_bundle.friction_factor_out,
        'inside_friction_factor_method_in': tube_bundle.friction_factor_method_in,
        'inside_friction_factor_method_mid': tube_bundle.friction_factor_method_mid,
        'inside_friction_factor_method_out': tube_bundle.friction_factor_method_out,
        'inside_dp_straight_tube_friction_Pa': tube_bundle.dp_straight_tube_friction,
        'inside_dp_straight_tube_acceleration_Pa': tube_bundle.dp_straight_tube_acceleration,
        'inside_dp_tube_entrances_Pa': tube_bundle.dp_tube_entrances,
        'inside_dp_tube_exits_Pa': tube_bundle.dp_tube_exits,
        'inside_dp_tube_bundle_Pa': tube_bundle.dp_tube_bundle,
    }


process_rows, thermal_rows, fluid_rows, error_rows = [], [], [], []

for case in cases:
    try:
        inside_provider = provider_for_medium(case['inside_medium'])
        outside_provider = provider_for_medium(case['outside_medium'])

        mode = case['mode']
        common = {
            'case': case['case'], 'mode': mode,
            'euler_provider': euler_provider,
            'inside_medium': case['inside_medium'],
            'outside_medium': case['outside_medium'],
        }

        if mode == 'simulate':
            overdesign = case.get('overdesign', 0.0)
            if overdesign is None:
                overdesign = 0.0
            inside = HXSideInput(
                provider=inside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['inside_m_dot_kg_h']),
                T_in=c_to_k(case['inside_T_in_C']),
            )
            outside = HXSideInput(
                provider=outside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['outside_m_dot_kg_h']),
                T_in=c_to_k(case['outside_T_in_C']),
            )
            result = hx.simulate(
                inside, outside, surface_margin=overdesign,
                euler_provider=euler_provider,
            )
            # Single source of truth: the converged thermal state that
            # simulate() itself consumed for inside_alfa_mean/U_mean/UA. No
            # separate solve_thermal_state() call is made here.
            state = result.thermal_state
            warning_codes = ', '.join(w.code for w in (result.warnings or []))
            process_rows.append({
                **common,
                'inside_m_dot_kg_h': inside.m_dot * 3600.0,
                'inside_T_in_C': inside.T_in - 273.15,
                'inside_T_out_C': result.T_out_inside - 273.15,
                'outside_m_dot_kg_h': outside.m_dot * 3600.0,
                'outside_T_in_C': outside.T_in - 273.15,
                'outside_T_out_C': result.T_out_outside - 273.15,
                'inside_velocity_m_s': result.inside_velocity_mean,
                'outside_velocity_m_s': result.outside_velocity_mean,
                'inside_dp_total_Pa': result.final_result.tube_side_hydraulic.dp_total,
                'inside_dp_friction_Pa': result.final_result.tube_side_hydraulic.dp_friction,
                'inside_dp_acceleration_Pa': result.final_result.tube_side_hydraulic.dp_acceleration,
                'inside_dp_bundle_Pa': result.final_result.tube_side_hydraulic.dp_tube_bundle,
                **tube_bundle_roughness_diagnostics(result.final_result.tube_side_hydraulic.tube_bundle),
                'inside_dp_local_Pa': result.inside_dp_local,
                'outside_dp_drag_Pa': result.outside_dp_drag,
                'outside_dp_acceleration_Pa': result.outside_dp_acceleration,
                'outside_dp_local_Pa': result.outside_dp_local,
                'outside_dp_total_Pa': result.outside_dp_total,
                'Q_required_kW': None, 'Q_achievable_kW': result.q / 1e3,
                'effectiveness_required': None, 'warnings': warning_codes,
            })
            thermal_rows.append({
                **common,
                'inside_Re': result.inside_Re_mean,
                'outside_Re': result.outside_Re_mean,
                'inside_alfa_W_m2K': result.inside_alfa_mean,
                'outside_alfa_W_m2K': result.outside_alfa_mean,
                'U_mean_W_m2K': result.U_mean, 'UA_actual_W_K': result.UA,
                'UA_effective_W_K': result.UA / (1.0 + overdesign),
                'UA_required_W_K': None, 'A_actual_m2': result.final_result.A_o,
                'A_required_m2': None,
                'overdesign_input_pct': 100.0 * overdesign,
                'overdesign_pct': 100.0 * result.overdesign_factor,
                'UA_margin_pct': None,
                **thermal_details_from_state(state, result),
            })
            fluid_rows.append({
                **common,
                'inside_rho_kg_m3': result.inside_props_mean.rho,
                'inside_mu_Pa_s': result.inside_props_mean.mu,
                'inside_k_W_mK': result.inside_props_mean.k,
                'inside_cp_J_kgK': result.inside_props_mean.cp,
                'outside_rho_kg_m3': result.outside_props_mean.rho,
                'outside_mu_Pa_s': result.outside_props_mean.mu,
                'outside_k_W_mK': result.outside_props_mean.k,
                'outside_cp_J_kgK': result.outside_props_mean.cp,
                **wall_props_from_state(state),
            })

        elif mode == 'rate':
            inside = BalanceSideSpec(
                provider=inside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['inside_m_dot_kg_h']),
                T_in=c_to_k(case['inside_T_in_C']), T_out=c_to_k(case['inside_T_out_C']),
            )
            outside = BalanceSideSpec(
                provider=outside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['outside_m_dot_kg_h']),
                T_in=c_to_k(case['outside_T_in_C']), T_out=c_to_k(case['outside_T_out_C']),
            )
            result = hx.rate(
                inside, outside, include_simulation=True,
                euler_provider=euler_provider, **rating_kwargs(case),
            )
            balance = result.closed_balance
            # Single source of truth: the converged thermal state that
            # rate() itself consumed for alfa_i/alfa_o/U_mean/UA_actual. No
            # separate solve_thermal_state() call is made here.
            state = result.thermal_state
            warning_codes = ', '.join(w.code for w in (result.warnings or []))

            # include_simulation=True only powers the velocity/Re/dp
            # snapshot below (not available on HXRatingResult itself);
            # alfa_i/alfa_o/fluid bulk properties come directly from
            # result/state, independent of this optional bridge.
            simulation = result.simulation
            if simulation is not None:
                inside_re = simulation.inside_Re_mean
                outside_re = simulation.outside_Re_mean
                inside_velocity = simulation.inside_velocity_mean
                outside_velocity = simulation.outside_velocity_mean
                inside_dp = simulation.final_result.tube_side_hydraulic.dp_total
                inside_dp_friction = simulation.final_result.tube_side_hydraulic.dp_friction
                inside_dp_acceleration = simulation.final_result.tube_side_hydraulic.dp_acceleration
                inside_dp_bundle = simulation.final_result.tube_side_hydraulic.dp_tube_bundle
                inside_dp_local = simulation.inside_dp_local
                tube_bundle = simulation.final_result.tube_side_hydraulic.tube_bundle
                outside_dp_drag = simulation.outside_dp_drag
                outside_dp_acceleration = simulation.outside_dp_acceleration
                outside_dp_local = simulation.outside_dp_local
                outside_dp = simulation.outside_dp_total
            else:
                inside_re = outside_re = float('nan')
                inside_velocity = outside_velocity = float('nan')
                inside_dp = inside_dp_friction = inside_dp_acceleration = inside_dp_bundle = inside_dp_local = outside_dp_drag = outside_dp_acceleration = outside_dp_local = outside_dp = float('nan')
                tube_bundle = None

            process_rows.append({
                **common,
                'inside_m_dot_kg_h': balance.inside.m_dot * 3600.0,
                'inside_T_in_C': balance.inside.T_in - 273.15,
                'inside_T_out_C': balance.inside.T_out - 273.15,
                'outside_m_dot_kg_h': balance.outside.m_dot * 3600.0,
                'outside_T_in_C': balance.outside.T_in - 273.15,
                'outside_T_out_C': balance.outside.T_out - 273.15,
                'inside_velocity_m_s': inside_velocity,
                'outside_velocity_m_s': outside_velocity,
                'inside_dp_total_Pa': inside_dp,
                'inside_dp_friction_Pa': inside_dp_friction,
                'inside_dp_acceleration_Pa': inside_dp_acceleration,
                'inside_dp_bundle_Pa': inside_dp_bundle,
                **tube_bundle_roughness_diagnostics(tube_bundle),
                'inside_dp_local_Pa': inside_dp_local,
                'outside_dp_drag_Pa': outside_dp_drag,
                'outside_dp_acceleration_Pa': outside_dp_acceleration,
                'outside_dp_local_Pa': outside_dp_local,
                'outside_dp_total_Pa': outside_dp,
                'Q_required_kW': result.Q_required / 1e3,
                'Q_achievable_kW': result.Q_achievable / 1e3,
                'effectiveness_required': balance.effectiveness,
                'warnings': warning_codes,
            })
            thermal_rows.append({
                **common,
                'inside_Re': inside_re,
                'outside_Re': outside_re,
                # Authoritative corrected coefficients straight from
                # HXRatingResult -- NOT from the simulation bridge.
                'inside_alfa_W_m2K': result.alfa_i,
                'outside_alfa_W_m2K': result.alfa_o,
                'U_mean_W_m2K': result.U_mean, 'UA_actual_W_K': result.UA_actual,
                'UA_effective_W_K': None, 'UA_required_W_K': result.UA_required,
                'A_actual_m2': result.A_o, 'A_required_m2': result.A_required,
                'overdesign_input_pct': None,
                'overdesign_pct': 100.0 * result.overdesign_factor,
                'UA_margin_pct': 100.0 * result.ua_margin,
                **thermal_details_from_state(state, result),
            })
            fluid_rows.append({
                **common,
                'inside_rho_kg_m3': state.inside_bulk_props.rho,
                'inside_mu_Pa_s': state.inside_bulk_props.mu,
                'inside_k_W_mK': state.inside_bulk_props.k,
                'inside_cp_J_kgK': state.inside_bulk_props.cp,
                'outside_rho_kg_m3': state.outside_bulk_props.rho,
                'outside_mu_Pa_s': state.outside_bulk_props.mu,
                'outside_k_W_mK': state.outside_bulk_props.k,
                'outside_cp_J_kgK': state.outside_bulk_props.cp,
                **wall_props_from_state(state),
            })
        else:
            raise ValueError(f"Nieobs\u0142ugiwany mode={mode!r}; u\u017cyj 'simulate' lub 'rate'.")
    except Exception as exc:
        error_rows.append({
            'case': case['case'], 'mode': case['mode'],
            'euler_provider': euler_provider, 'error': str(exc),
        })

process_results = pd.DataFrame(process_rows)
thermal_results = pd.DataFrame(thermal_rows)
fluid_results = pd.DataFrame(fluid_rows)
errors = pd.DataFrame(error_rows)
assert errors.empty, errors.to_dict('records')
assert len(process_results) == len(cases) == len(thermal_results) == len(fluid_results)

# Authoritative-source assertions (v0.5.3): the exported "public" alfa
# column must equal the corrected diagnostic value on every row -- this is
# the notebook-level guard against ever re-introducing a second,
# independent calculation of the corrected coefficient.
import math as _math

for _, _row in thermal_results.iterrows():
    if _math.isnan(_row['inside_alfa_corrected']):
        continue  # iterate=False rows, if any, carry no thermal_state
    assert _math.isclose(
        _row['inside_alfa_W_m2K'], _row['inside_alfa_corrected'],
        rel_tol=1e-9, abs_tol=1e-12,
    ), _row.to_dict()


## Thermal process results

In [5]:
process_table = process_results.round({
    'inside_m_dot_kg_h': 1, 'inside_T_in_C': 2, 'inside_T_out_C': 2,
    'outside_m_dot_kg_h': 1, 'outside_T_in_C': 2, 'outside_T_out_C': 2,
    'inside_velocity_m_s': 3, 'outside_velocity_m_s': 3,
    'inside_dp_total_Pa': 1, 'inside_dp_local_Pa': 1, 'outside_dp_drag_Pa': 1,
    'outside_dp_acceleration_Pa': 1, 'outside_dp_local_Pa': 1, 'outside_dp_total_Pa': 1,
    'inside_roughness_inner_mm': 3, 'inside_relative_roughness': 6,
    'inside_friction_factor_in': 5, 'inside_friction_factor_mid': 5, 'inside_friction_factor_out': 5,
    'inside_dp_straight_tube_friction_Pa': 1, 'inside_dp_straight_tube_acceleration_Pa': 1,
    'inside_dp_tube_entrances_Pa': 1, 'inside_dp_tube_exits_Pa': 1, 'inside_dp_tube_bundle_Pa': 1,
    'Q_required_kW': 2, 'Q_achievable_kW': 2, 'effectiveness_required': 4,
})
thermal_table = thermal_results.round({
    'inside_Re': 0, 'outside_Re': 0,
    'inside_alfa_W_m2K': 2, 'outside_alfa_W_m2K': 2,
    'U_mean_W_m2K': 2, 'UA_actual_W_K': 1, 'UA_effective_W_K': 1,
    'UA_required_W_K': 1,
    'A_actual_m2': 2, 'A_required_m2': 2,
    'overdesign_input_pct': 2, 'overdesign_pct': 2, 'UA_margin_pct': 2,
    'inside_bulk_temperature': 2,
    'inside_wall_temperature': 2,
    'inside_wall_temperature_mean': 2,
    'inside_wall_temperature_min_estimate': 2,
    'inside_wall_temperature_max_estimate': 2,
    'outside_wall_temperature_mean': 2,
    'outside_wall_temperature_min_estimate': 2,
    'outside_wall_temperature_max_estimate': 2,
    'inside_Nu_base': 3,
    'inside_length_correction': 6,
    'inside_wall_temperature_correction': 6,
    'inside_Nu_corrected': 3,
    'inside_alfa_base': 2,
    'inside_alfa_corrected': 2,
})
fluid_table = fluid_results.round({
    'inside_rho_kg_m3': 4, 'outside_rho_kg_m3': 4,
    'inside_mu_Pa_s': 8, 'outside_mu_Pa_s': 8,
    'inside_k_W_mK': 5, 'outside_k_W_mK': 5,
    'inside_cp_J_kgK': 2, 'outside_cp_J_kgK': 2,
    'inside_wall_mu_Pa_s': 8, 'outside_wall_mu_Pa_s': 8,
    'inside_wall_k_W_mK': 5, 'outside_wall_k_W_mK': 5,
    'inside_wall_cp_J_kgK': 2, 'outside_wall_cp_J_kgK': 2,
    'inside_wall_Pr': 4, 'outside_wall_Pr': 4,
})

display(process_table)
display(thermal_table)
display(fluid_table)


,case,mode,euler_provider,inside_medium,outside_medium,inside_m_dot_kg_h,inside_T_in_C,inside_T_out_C,outside_m_dot_kg_h,outside_T_in_C,...,inside_dp_tube_bundle_Pa,inside_dp_local_Pa,outside_dp_drag_Pa,outside_dp_acceleration_Pa,outside_dp_local_Pa,outside_dp_total_Pa,Q_required_kW,Q_achievable_kW,effectiveness_required,warnings
0,S01,simulate,gaddis_gnielinski,dry_air,wet_gas_mixture,18220.0,30.0,187.78,28380.0,400.0,...,116.2,0.0,735.3,-1.0,0.0,734.3,NaN,808.81,NaN,tube_bundle_hydraulics_midpoint_temperature_fa...
1,R01,rate,gaddis_gnielinski,dry_air,wet_gas_mixture,18220.0,30.0,195.51,28380.0,400.0,...,119.9,0.0,730.5,-1.1,0.0,729.4,848.15,878.55,0.4473,tube_bundle_hydraulics_midpoint_temperature_fa...


,case,mode,euler_provider,inside_medium,outside_medium,inside_Re,outside_Re,inside_alfa_W_m2K,outside_alfa_W_m2K,U_mean_W_m2K,...,inside_wall_temperature_max_estimate,outside_wall_temperature_mean,outside_wall_temperature_min_estimate,outside_wall_temperature_max_estimate,inside_Nu_base,inside_length_correction,inside_wall_temperature_correction,inside_Nu_corrected,inside_alfa_base,inside_alfa_corrected
0,S01,simulate,gaddis_gnielinski,dry_air,wet_gas_mixture,10252.0,11833.0,25.3,110.57,19.1,...,634.98,584.64,540.49,635.13,30.406,1.063987,0.832421,26.93,28.57,25.3
1,R01,rate,gaddis_gnielinski,dry_air,wet_gas_mixture,10252.0,11833.0,25.3,110.57,19.1,...,636.06,584.64,536.70,636.20,30.406,1.063987,0.832421,26.93,28.57,25.3


,case,mode,euler_provider,inside_medium,outside_medium,inside_rho_kg_m3,inside_mu_Pa_s,inside_k_W_mK,inside_cp_J_kgK,outside_rho_kg_m3,...,outside_k_W_mK,outside_cp_J_kgK,inside_wall_mu_Pa_s,inside_wall_k_W_mK,inside_wall_cp_J_kgK,inside_wall_Pr,outside_wall_mu_Pa_s,outside_wall_k_W_mK,outside_wall_cp_J_kgK,outside_wall_Pr
0,S01,simulate,gaddis_gnielinski,dry_air,wet_gas_mixture,0.9077,0.000023,0.0327,1012.86,0.5279,...,0.04727,1168.37,0.00003,0.04509,1047.66,0.702,0.000028,0.04458,1157.1,0.735
1,R01,rate,gaddis_gnielinski,dry_air,wet_gas_mixture,0.9077,0.000023,0.0327,1012.86,0.5279,...,0.04727,1168.37,0.00003,0.04509,1047.66,0.702,0.000028,0.04458,1157.1,0.735


In [6]:
# export_dir = workspace_root / 'core' / 'tests' / 'outputs'
# export_dir.mkdir(parents=True, exist_ok=True)
# export_path = export_dir / 'heat_balance_rating_matrix_results.xlsx'
# with pd.ExcelWriter(export_path) as writer:
#     process_table.to_excel(writer, sheet_name='process', index=False)
#     thermal_table.to_excel(writer, sheet_name='thermal', index=False)
#     fluid_table.to_excel(writer, sheet_name='fluid_props', index=False)
# print(f'Zapisano plik Excel: {export_path}')

## Inlet, midpoint, and outlet fluid properties

Point states below come directly from the solver's existing hydraulic results. The wet midpoint uses arithmetic mean temperature and water ratio; no transport-property provider is called for presentation.

In [7]:
import math
import pandas as pd


def endpoint_property_table(solver_result, side):
    """Read solver-owned hydraulic point states without provider calls."""
    states = (
        ("inlet", getattr(solver_result, f"{side}_properties_inlet")),
        ("midpoint", getattr(solver_result, f"{side}_properties_midpoint")),
        ("outlet", getattr(solver_result, f"{side}_properties_outlet")),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "T [°C]": state.T - 273.15,
                "p [Pa]": state.p,
                "rho [kg/m³]": state.rho,
                "cp [J/(kg·K)]": state.cp,
                "mu [Pa·s]": state.mu,
                "k [W/(m·K)]": state.k,
                "Pr [-]": state.Pr,
            }
            for name, state in states
            if state is not None
        ]
    ).set_index("state")


def representative_0d_property_table(solver_result):
    """Keep representative thermal properties separate from point states."""
    if hasattr(solver_result, "inside_props_mean"):
        pairs = (
            ("inside", solver_result.T_mean_inside, solver_result.inside_props_mean),
            ("outside", solver_result.T_mean_outside, solver_result.outside_props_mean),
        )
    elif getattr(solver_result, "thermal_state", None) is not None:
        thermal = solver_result.thermal_state
        pairs = (
            ("inside", thermal.inside_bulk_temperature, thermal.inside_bulk_props),
            ("outside", thermal.outside_bulk_temperature, thermal.outside_bulk_props),
        )
    else:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "side": side,
                "T representative [°C]": temperature - 273.15,
                "rho [kg/m³]": props.rho,
                "cp [J/(kg·K)]": props.cp,
                "mu [Pa·s]": props.mu,
                "k [W/(m·K)]": props.k,
                "Pr [-]": props.mu * props.cp / props.k,
            }
            for side, temperature, props in pairs
        ]
    ).set_index("side")


def wet_gas_state_table(solver_result, outside_provider_for_result=None):
    """Combine hydraulic states with wet diagnostics already returned by the solver."""
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable or pc.W_in is None:
        return pd.DataFrame()

    W_mid = 0.5 * (pc.W_in + pc.W_out)
    dew_mid = math.nan
    if outside_provider_for_result is not None:
        from core.phase_change.capability import detect_phase_change_capability
        from core.phase_change.integration import _dew_point_at_ratio

        capability = detect_phase_change_capability(outside_provider_for_result)
        midpoint_state = solver_result.outside_properties_midpoint
        dew_mid_value = _dew_point_at_ratio(
            capability, W_mid, p=midpoint_state.p
        )
        dew_mid = math.nan if dew_mid_value is None else dew_mid_value

    dry_flow = pc.m_dot_dry_carrier
    vapor_mid = (
        math.nan
        if dry_flow is None
        else dry_flow * W_mid
    )
    gas_mid = (
        math.nan
        if dry_flow is None
        else dry_flow + vapor_mid
    )
    values = (
        ("inlet", pc.W_in, pc.dew_point_in, pc.m_dot_gas_in, pc.m_dot_water_vapor_in),
        ("midpoint", W_mid, dew_mid, gas_mid, vapor_mid),
        ("outlet", pc.W_out, pc.dew_point_out, pc.m_dot_gas_out, pc.m_dot_water_vapor_out),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "W [kg/kg dry]": W,
                "dew point [°C]": (
                    math.nan if dew_point is None else dew_point - 273.15
                ),
                "m_dot gas [kg/s]": gas_flow,
                "m_dot water vapor [kg/s]": vapor_flow,
            }
            for name, W, dew_point, gas_flow, vapor_flow in values
        ]
    ).set_index("state")


def condensation_summary_table(solver_result):
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable:
        return pd.DataFrame()

    def _to_c(value):
        return math.nan if value is None else value - 273.15

    W_bulk = (
        math.nan
        if pc.W_in is None or pc.W_out is None
        else 0.5 * (pc.W_in + pc.W_out)
    )
    return pd.DataFrame(
        [
            {
                "T_dew_in [degC]": _to_c(pc.dew_point_in),
                "T_dew_out [degC]": _to_c(pc.dew_point_out),
                "T_wall_min [degC]": _to_c(pc.wall_temperature_min),
                "T_wall_mean [degC]": _to_c(pc.wall_temperature_mean),
                "T_wall_max [degC]": _to_c(pc.wall_temperature_max),
                "T_wall_wet_mean [degC]": _to_c(pc.wall_temperature_wet_mean),
                "wet_surface_fraction [-]": pc.wet_surface_fraction,
                "A_wet [m2]": pc.wet_area,
                "A_outside [m2]": pc.outside_total_area,
                "W_bulk representative [kg/kg dry]": W_bulk,
                "W_sat_wet_surface [kg/kg dry]": pc.W_sat_wet_surface,
                "W_in [kg/kg dry]": pc.W_in,
                "W_out [kg/kg dry]": pc.W_out,
                "m_dot condensate [kg/s]": pc.m_dot_condensate,
                "Q_sensible [W]": pc.Q_sensible,
                "Q_latent [W]": pc.Q_latent,
                "Q_total [W]": pc.Q_total,
                "alfa_dry [W/(m2 K)]": pc.alfa_dry,
                "alfa_effective [W/(m2 K)]": pc.alfa_effective,
            }
        ],
        index=["outside"],
    )

In [8]:
endpoint_results = [('Final matrix result', result)]
outside_provider_for_endpoint_table = globals().get('outside_provider')

for result_label, endpoint_result in endpoint_results:
    print(result_label)
    print("Inside")
    display(endpoint_property_table(endpoint_result, "inside"))
    print("Outside")
    display(endpoint_property_table(endpoint_result, "outside"))

    wet_table = wet_gas_state_table(
        endpoint_result, outside_provider_for_endpoint_table
    )
    if not wet_table.empty:
        print("Outside wet-gas mass and dew-point diagnostics")
        display(wet_table)
        display(condensation_summary_table(endpoint_result))

Final matrix result
Inside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,30.000000,101325.0,1.164734,1006.492185,0.000019,0.026618,0.706669
midpoint,112.752704,101325.0,0.914574,1012.540619,0.000022,0.032496,0.699562
outlet,195.505408,101325.0,0.752966,1024.189010,0.000026,0.037962,0.697929


Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,400.000000,101325.0,0.490531,1181.916787,0.000032,0.050404,0.739017
midpoint,354.230644,101325.0,0.526323,1168.902410,0.000030,0.047400,0.736916
outlet,307.953160,101325.0,0.568250,1156.149845,0.000028,0.044341,0.734883


Outside wet-gas mass and dew-point diagnostics


,W [kg/kg dry],dew point [°C],m_dot gas [kg/s],m_dot water vapor [kg/s]
state,,,,
inlet,0.120005,55.74619,7.883333,0.844676
midpoint,0.120005,55.74619,7.883333,0.844676
outlet,0.120005,55.74619,7.883333,0.844676


,T_dew_in [degC],T_dew_out [degC],T_wall_min [degC],T_wall_mean [degC],T_wall_max [degC],T_wall_wet_mean [degC],wet_surface_fraction [-],A_wet [m2],A_outside [m2],W_bulk representative [kg/kg dry],W_sat_wet_surface [kg/kg dry],W_in [kg/kg dry],W_out [kg/kg dry],m_dot condensate [kg/s],Q_sensible [W],Q_latent [W],Q_total [W],alfa_dry [W/(m2 K)],alfa_effective [W/(m2 K)]
outside,55.74619,55.74619,263.549888,311.485433,363.051284,NaN,None,None,None,0.120005,None,0.120005,0.120005,0.0,0.0,0.0,0.0,None,None


## Representative 0D properties used by the solver

These lumped thermal-model properties are retained separately; they are not substitutes for inlet or outlet states.

In [9]:
for result_label, endpoint_result in endpoint_results:
    representative_table = representative_0d_property_table(endpoint_result)
    if not representative_table.empty:
        print(result_label)
        display(representative_table)

Final matrix result


,T representative [°C],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
side,,,,,,
inside,115.674387,0.907694,1012.859372,0.000023,0.032695,0.699419
outside,352.318992,0.527932,1168.365742,0.000030,0.047275,0.736829
